In [1]:
from Value import Value
from NN import NN
from graph import draw_graph

In [2]:
# MNIST classifier
from tensorflow.keras.datasets import mnist

# Y is the labels, x is the images
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Number of images to train on
n_train = 1000
# Number of training loops
epochs = 100

x_train = x_train[:n_train]
y_train = y_train[:n_train]
dataset = x_train.reshape(n_train, -1) / 255.0
# Convert to Value objects
dataset = [[Value(pixel) for pixel in image] for image in dataset]
labels = y_train

# 3 layers with 16 neurons each
mlp = NN(784, [16, 16, 16, 10])
# Learning rate for SGD
lr = .01

In [3]:
# Training loop
for i in range(epochs):
    avg_loss = 0

    for j in range(n_train):
        # Zero gradients
        for param in mlp.get_params():
            param.grad = 0

        # Forward pass
        logits = mlp(dataset[j])

        # Normalize logits to probabilities
        denom = sum(logit.exp() for logit in logits)

        probs = [logit.exp() / denom for logit in logits]

        for k, prob in enumerate(probs):
            prob._op = "softmax"
            prob.label = f"prob_{k}"

        # Get cross-entropy loss
        loss = probs[labels[j]].log() * -1
        avg_loss += loss.v

        # Backward pass
        loss.backward()
        mlp.grad_descent(lr)
    
    avg_loss = avg_loss / n_train
    print(f"Epoch {i + 1},   Loss: {avg_loss:.6f}")

Epoch 1,   Loss: 2.412399
Epoch 2,   Loss: 2.159625


KeyboardInterrupt: 

In [ ]:
import pickle
with open('model.pkl', 'wb') as f:
    pickle.dump(mlp, f)

# trained for 2 epochs so far

In [ ]:
import pickle
with open('model.pkl', 'rb') as f:
    mlp = pickle.load(f)

In [15]:
# Testing the model
import numpy as np

# Number of images to test on
n_test = 2000

x_test = x_test[:n_test]
y_test = y_test[:n_test]
testset = x_test.reshape(n_test, -1) / 255.0
# Convert to Value objects
testset = [[Value(pixel) for pixel in image] for image in testset]
labels = y_test

n_correct = 0

for i in range(n_test):
    # Forward pass
    logits = mlp(testset[i])
    # Use scalar values instead of Values objects for speed
    logit_values = np.array([l.v for l in logits])

    # Normalize logits to probabilities and find top prob
    probs = np.exp(logit_values) / np.sum(np.exp(logit_values))    
    prediction = np.argmax(probs)

    if prediction == labels[i]: n_correct += 1

print(f"Accuracy: {(n_correct/n_test * 100):.2f}%")
    

Accuracy: 32.35%
